# 批量萜合酶序列核实工具

本工具用于批量处理大量序列文件，核实它们是否为萜合酶。

**功能特点：**
1. **自动扫描目录** - 自动发现所有序列文件（.fasta, .fa, .faa）
2. **结构文件关联** - 自动匹配对应的PDB结构文件（如 *_alphafold.pdb）
3. **批量验证** - 高效处理大量序列文件
4. **详细报告** - 生成每个文件的验证结果和汇总统计

**输入：** 包含序列文件的目录（如 `A0A0B4G3Q7.fasta`, `A0A0B4G3Q7_alphafold.pdb`）
**输出：** 验证结果CSV文件、统计报告、分类后的序列文件

## 1. 初始化环境

In [ ]:
from protflow.utils.notebook_utils import init_notebook
from pathlib import Path
from Bio import SeqIO
import pandas as pd

# 自动初始化环境
paths = init_notebook('batch_terpene_validation')
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']
INPUTS_DIR = paths.get('INPUTS_DIR', DATA_DIR / 'inputs')

print(f"✓ 工作目录: {WORK_DIR}")
print(f"✓ 数据目录: {DATA_DIR}")
print(f"✓ 输入目录: {INPUTS_DIR}")

## 2. 导入验证模块

In [ ]:
from protflow.utils.terpene_synthase_validator import (
    batch_validate_directory,
    validate_fasta_file
)

print("✓ 验证模块导入成功")

## 3. 指定输入目录

指定包含序列文件的目录。工具会自动扫描所有 `.fasta`, `.fa`, `.faa` 文件。

**文件命名示例：**
- `A0A0B4G3Q7.fasta` - 序列文件
- `A0A0B4G3Q7_alphafold.pdb` - 对应的结构文件（可选）

In [ ]:
# 指定输入目录（请修改为您的实际目录路径）
# 方式1: 使用统一的 inputs 目录
batch_input_dir = INPUTS_DIR / 'terpene_synthase' / 'batch'
batch_input_dir.mkdir(exist_ok=True, parents=True)

# 方式2: 直接指定目录路径（如果文件在其他位置）
# batch_input_dir = Path('d:/Projects/ProtFlow/待办')
# batch_input_dir = Path('path/to/your/sequences')

if not batch_input_dir.exists():
    print(f"⚠️ 输入目录不存在: {batch_input_dir}")
    print(f"\n请将序列文件放在以下目录：")
    print(f"  {batch_input_dir}")
    print(f"\n或修改上面的 batch_input_dir 路径")
else:
    # 扫描序列文件
    fasta_files = list(batch_input_dir.glob('*.fasta'))
    fasta_files.extend(batch_input_dir.glob('*.fa'))
    fasta_files.extend(batch_input_dir.glob('*.faa'))
    fasta_files = sorted(list(set(fasta_files)), key=lambda x: x.name)
    
    # 扫描PDB文件
    pdb_files = list(batch_input_dir.glob('*.pdb'))
    pdb_files = sorted(list(set(pdb_files)), key=lambda x: x.name)
    
    print(f"✓ 输入目录: {batch_input_dir}")
    print(f"  序列文件数: {len(fasta_files)}")
    print(f"  结构文件数: {len(pdb_files)}")
    
    if len(fasta_files) > 0:
        print(f"\n前10个序列文件：")
        for i, f in enumerate(fasta_files[:10], 1):
            # 检查是否有对应的PDB文件
            file_id = f.stem
            has_pdb = any(file_id in pdb.name for pdb in pdb_files)
            pdb_marker = "📦" if has_pdb else "  "
            print(f"  {i}. {pdb_marker} {f.name}")
        
        if len(fasta_files) > 10:
            print(f"  ... 还有 {len(fasta_files) - 10} 个文件")
    else:
        print(f"\n⚠️ 未找到序列文件，请检查目录路径")

## 4. 配置批量验证参数

In [ ]:
# 验证方法配置
USE_KEYWORDS = True   # 使用关键词搜索
USE_PROSITE = True    # 使用PROSITE模式匹配
USE_FEATURES = True   # 使用序列特征分析

# 文件匹配配置
MATCH_PDB = True      # 是否尝试匹配对应的PDB文件
PDB_PATTERN = "*_alphafold.pdb"  # PDB文件匹配模式

# 输出配置
output_dir = WORK_DIR / 'batch_validation_results'
output_dir.mkdir(exist_ok=True, parents=True)

print("批量验证配置：")
print(f"  关键词搜索: {'启用' if USE_KEYWORDS else '禁用'}")
print(f"  PROSITE模式: {'启用' if USE_PROSITE else '禁用'}")
print(f"  序列特征分析: {'启用' if USE_FEATURES else '禁用'}")
print(f"  匹配PDB文件: {'启用' if MATCH_PDB else '禁用'}")
print(f"\n输出目录: {output_dir}")

## 5. 运行批量验证

In [ ]:
if not batch_input_dir.exists() or len(fasta_files) == 0:
    print("⚠️ 请先设置正确的输入目录并确保有序列文件（第3步）")
else:
    try:
        print("\n开始批量验证序列...")
        print("=" * 60)
        
        # 运行批量验证
        all_results, batch_stats = batch_validate_directory(
            input_dir=batch_input_dir,
            pattern="*.fasta",
            output_dir=output_dir,
            use_keywords=USE_KEYWORDS,
            use_prosite=USE_PROSITE,
            use_features=USE_FEATURES,
            match_pdb=MATCH_PDB,
            pdb_pattern=PDB_PATTERN
        )
        
        print("\n" + "=" * 60)
        print("批量验证完成！")
        
    except Exception as e:
        print(f"\n✗ 批量验证失败: {e}")
        import traceback
        traceback.print_exc()

## 6. 查看批量验证结果

In [ ]:
if 'all_results' in locals() and 'batch_stats' in locals():
    # 显示统计摘要
    print("=" * 60)
    print("批量验证结果统计")
    print("=" * 60)
    print(f"处理文件数: {batch_stats['processed']}/{batch_stats['total_files']}")
    if batch_stats['failed'] > 0:
        print(f"失败文件数: {batch_stats['failed']}")
    print(f"\n总序列数: {batch_stats['total_sequences']}")
    print(f"\n萜合酶序列: {batch_stats['terpene_synthase_sequences']} ({batch_stats['terpene_synthase_sequences']/batch_stats['total_sequences']*100:.1f}%)")
    print(f"  高置信度: {batch_stats['high_confidence']}")
    print(f"  中置信度: {batch_stats['medium_confidence']}")
    print(f"  低置信度: {batch_stats['low_confidence']}")
    print(f"\n非萜合酶序列: {batch_stats['not_terpene_synthase']} ({batch_stats['not_terpene_synthase']/batch_stats['total_sequences']*100:.1f}%)")
    print(f"\n有结构文件的序列: {batch_stats['files_with_pdb']}")
    
    # 加载汇总结果
    summary_csv = output_dir / 'batch_validation_summary.csv'
    if summary_csv.exists():
        df = pd.read_csv(summary_csv)
        
        # 按文件分组统计
        file_summary = df.groupby('source_file').agg({
            'is_terpene_synthase': ['sum', 'count'],
            'confidence': lambda x: (x == 'high').sum(),
            'has_structure': 'any'
        }).reset_index()
        file_summary.columns = ['file', 'terpene_count', 'total_count', 'high_conf_count', 'has_structure']
        file_summary['terpene_ratio'] = file_summary['terpene_count'] / file_summary['total_count']
        
        print(f"\n按文件统计（前10个文件）：")
        print(file_summary.head(10).to_string(index=False))
        
        print(f"\n详细结果已保存到: {summary_csv}")
    else:
        print("⚠️ 汇总结果文件未找到")
else:
    print("⚠️ 请先运行批量验证（第5步）")

## 7. 导出分类结果（可选）

In [ ]:
if 'all_results' in locals() and 'df' in locals():
    # 创建分类输出目录
    classified_dir = output_dir / 'classified_sequences'
    classified_dir.mkdir(exist_ok=True, parents=True)
    
    # 高置信度萜合酶序列
    high_conf_ids = df[(df['is_terpene_synthase'] == True) & (df['confidence'] == 'high')]['id'].tolist()
    
    # 所有萜合酶序列
    all_ts_ids = df[df['is_terpene_synthase'] == True]['id'].tolist()
    
    # 非萜合酶序列
    non_ts_ids = df[df['is_terpene_synthase'] == False]['id'].tolist()
    
    # 从原始文件提取并分类保存
    high_conf_seqs = []
    all_ts_seqs = []
    non_ts_seqs = []
    
    # 从所有源文件中提取序列
    source_files = df['source_file'].unique()
    for source_file in source_files:
        source_path = batch_input_dir / source_file
        if source_path.exists():
            for seq in SeqIO.parse(source_path, 'fasta'):
                if seq.id in high_conf_ids:
                    high_conf_seqs.append(seq)
                if seq.id in all_ts_ids:
                    all_ts_seqs.append(seq)
                if seq.id in non_ts_ids:
                    non_ts_seqs.append(seq)
    
    # 保存分类结果
    if high_conf_seqs:
        high_conf_fasta = classified_dir / 'high_confidence_terpene_synthases.faa'
        SeqIO.write(high_conf_seqs, high_conf_fasta, 'fasta')
        print(f"✓ 高置信度萜合酶序列: {len(high_conf_seqs)} 条")
        print(f"  保存到: {high_conf_fasta}")
    
    if all_ts_seqs:
        all_ts_fasta = classified_dir / 'all_terpene_synthases.faa'
        SeqIO.write(all_ts_seqs, all_ts_fasta, 'fasta')
        print(f"\n✓ 所有萜合酶序列: {len(all_ts_seqs)} 条")
        print(f"  保存到: {all_ts_fasta}")
    
    if non_ts_seqs:
        non_ts_fasta = classified_dir / 'non_terpene_synthases.faa'
        SeqIO.write(non_ts_seqs, non_ts_fasta, 'fasta')
        print(f"\n✓ 非萜合酶序列: {len(non_ts_seqs)} 条")
        print(f"  保存到: {non_ts_fasta}")
    
    # 如果有结构文件，也进行分类
    if MATCH_PDB:
        pdb_classified_dir = classified_dir / 'structures'
        pdb_classified_dir.mkdir(exist_ok=True, parents=True)
        
        # 高置信度萜合酶的结构文件
        high_conf_pdb = df[(df['is_terpene_synthase'] == True) & (df['confidence'] == 'high') & (df['has_structure'] == True)]
        if len(high_conf_pdb) > 0:
            pdb_high_dir = pdb_classified_dir / 'high_confidence'
            pdb_high_dir.mkdir(exist_ok=True)
            
            for _, row in high_conf_pdb.iterrows():
                if pd.notna(row['pdb_path']):
                    pdb_src = Path(row['pdb_path'])
                    if pdb_src.exists():
                        import shutil
                        shutil.copy2(pdb_src, pdb_high_dir / pdb_src.name)
            
            copied_pdb = len(list(pdb_high_dir.glob('*.pdb')))
            print(f"\n✓ 高置信度萜合酶结构文件: {copied_pdb} 个")
            print(f"  保存到: {pdb_high_dir}")
else:
    print("⚠️ 请先运行批量验证（第5步）")